<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_01_4_genai_deep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# T81-558: Applications of Deep Neural Networks
**Module 1: Introduction to Neural Networks**
* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 1 Material

* Part 1.1: Course Overview [[Video]]() [[Notebook]](t81_558_class_01_1_overview.ipynb)
* Part 1.2: AI and Neural Networks [[Video]]() [[Notebook]](t81_558_class_01_2_ai_neural.ipynb)
* Part 1.3: Modern Neural Networks [[Video]]() [[Notebook]](t81_558_class_01_3_neural_net.ipynb)
* **Part 1.4: Deep Learning in the Age of GenAI** [[Video]]() [[Notebook]](t81_558_class_01_4_genai_deep.ipynb)
* Part 1.5: Vibe Coding: Acceptable AI Use in this Course [[Video]]() [[Notebook]](t81_558_class_01_5_vibe_code.ipynb)

# Google CoLab Instructions

The following code ensures that Google CoLab is running and maps Google Drive if needed.


In [ ]:
try:
    import google.colab

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Part 1.4: Deep Learning in the Age of GenAI

PyTorch can implement a neural network for nearly any supervised or unsupervised learning task. The relevant engineering question is not whether PyTorch *can* solve a problem, but whether it is the right tool. The framework's flexibility is a liability as much as an asset: deploying a deep network where XGBoost would suffice wastes compute and interpretability; ignoring a pre-trained foundation model in favor of a custom architecture wastes months of training and data collection. Mapping a problem to the right tool class is the first design decision, and it has larger consequences than any subsequent hyperparameter choice.

The space of problems amenable to neural networks partitions cleanly into three categories, each with a different engineering rationale and a different recommended approach.

### When Classical ML Is the Better Fit

Gradient-boosted tree ensembles and other classical methods consistently outperform neural networks on a well-defined class of problems. The temptation to reach for PyTorch on any prediction task should be weighed against the cost of training data, interpretability requirements, and the empirical record on structured inputs.

- Tabular data with well-engineered features and under ~1M rows, where XGBoost or LightGBM consistently match or exceed neural network performance at a fraction of the training cost
- Problems with hard interpretability or audit requirements, where SHAP values on a tree ensemble satisfy compliance constraints that a deep network cannot
- Low-data regimes where a neural network will overfit without extensive regularization, and a simpler model generalizes better by default

### Where Foundation Models Have Displaced Custom Solutions

Several problem classes that drove neural network adoption over the past decade are now better handled by large pre-trained models accessed via fine-tuning or prompting. Building a custom architecture from scratch for these tasks is rarely justified: the pre-trained model arrives with representations learned from orders of magnitude more data than any single project can supply.

- Text classification, named entity recognition, summarization, and translation, where fine-tuning a transformer (BERT, T5, LLaMA) outperforms a trained-from-scratch RNN or CNN at lower cost and with less data
- Image synthesis and style transfer, where diffusion models and latent diffusion architectures (Stable Diffusion, DALL-E) produce results that GAN-based pipelines cannot match in fidelity or diversity. A GAN (Generative Adversarial Network) pits a generator network against a discriminator in a minimax game; diffusion models instead learn to reverse a gradual noising process, a formulation that proves more stable and scalable.
- Conditional generation and data augmentation, where GAN training instability and mode collapse made the approach fragile; diffusion models handle the same tasks more reliably

### The Current Sweet Spot for Custom PyTorch Networks

A well-defined class of problems still demands a custom-trained network: tasks requiring structured geometric output, tight latency constraints, or input modalities that no general-purpose foundation model currently handles. These problems share the property that routing inputs through a remote API is either too slow, too costly, or architecturally impossible.

- Real-time multi-object detection and localization, where architectures like YOLO (You Only Look Once) output class labels, bounding box coordinates, and confidence scores for dozens of objects in a single forward pass over the full image. Earlier detection pipelines used a separate region-proposal stage; YOLO collapses this into one network evaluated once per frame, enabling real-time throughput.
- Facial landmark detection, where the network must regress the precise $(x, y)$ coordinates of dozens of anatomical keypoints (eye corners, lip contour, nose tip) simultaneously. Applications include gaze estimation, driver monitoring, and AR face overlays, all of which require sub-frame latency and coordinates rather than a classification label.
- Domain-specific sensor fusion: combining LiDAR point clouds with camera frames for autonomous navigation, where no general-purpose model exists and the input modalities require custom architecture
- Lightweight on-device inference models trained via knowledge distillation, where size and latency budgets preclude using a large foundation model. Knowledge distillation trains a small "student" network to reproduce the output distribution of a larger "teacher," recovering most of the teacher's accuracy at a fraction of the parameter count.

A tabular fraud-detection pipeline will train in minutes with XGBoost and produce auditable feature importances; the equivalent neural network requires more data, more tuning, and outputs that are harder to explain to a regulator. A GAN trained in 2019 to generate synthetic faces is now strictly dominated by a diffusion model that converges more reliably and samples at higher resolution. The remaining territory—real-time coordinate regression, novel sensor modalities, on-device deployment—is where the investment of building and training a custom PyTorch network pays off. The three sections that follow develop each category with worked examples and implementation guidance.

